In [ ]:
import os
from pathlib import Path
DATA = Path(os.environ.get('ENSO_DATA_ROOT', Path.cwd() / 'data'))

import sys
# 获取当前作用域的所有变量名称
variable_names = list(globals().keys())
# 删除除内置变量和导入的模块以外的所有变量
for var in variable_names:
    if not var.startswith("__") and var not in sys.modules and var not in {"DATA", "os", "Path"}:
        del globals()[var]


In [ ]:
# 导入库
import scipy.io as sio
import numpy as np
import pandas as pd
from scipy.stats import chi2
import statsmodels.stats.weightstats as sw
from scipy.stats import norm
import netCDF4 as nc
import colormaps as cmaps  # 确保 colormaps 库已安装

In [ ]:
from scipy.interpolate import griddata
stdexp = sio.loadmat(str(DATA / "meansstd24.mat"))['std24']
lonstd= sio.loadmat(str(DATA / "meansstd24.mat"))['lona'].flatten()
stdexp = np.column_stack((stdexp, stdexp[:, 0]))
y = np.linspace(0, 360, 25)


In [ ]:
from scipy.interpolate import griddata
stdexp = sio.loadmat(str(DATA / "meansstd24_intep.mat"))['std24_intep']
lonstd= sio.loadmat(str(DATA / "meansstd24_intep.mat"))['lona_intep'].flatten()
y_interp= sio.loadmat(str(DATA / "meansstd24_intep.mat"))['y_interp'].flatten()
y = np.linspace(0, 360, 25)


In [ ]:

# --- 筛选经度范围 (NINO3: 270–330E) ---
idx_lon = np.where((lonstd > 270) & (lonstd <= 330))[0]

# --- 筛选 precession phase 范围 ---
idx_max = np.where((y_interp > 270) & (y_interp < 280))[0]   # Pmax
idx_min = np.where((y_interp > 90) & (y_interp <100 ))[0]    # Pmin

# --- 提取子集 ---
Pmax_std = stdexp[np.ix_(idx_max, idx_lon)]
Pmin_std = stdexp[np.ix_(idx_min, idx_lon)]

# --- 定义忽略 NaN 的平均与置信区间 ---
def mean_ci95(data):
    flat = data.flatten()
    flat = flat[~np.isnan(flat)]   # 去掉 NaN
    mean = np.nanmean(flat)
    std_err = np.nanstd(flat, ddof=1) / np.sqrt(len(flat))
    ci95 = 1.96 * std_err
    return mean, ci95

# --- 区域平均 + CI ---
Pmax_mean, Pmax_ci = mean_ci95(Pmax_std)
Pmin_mean, Pmin_ci = mean_ci95(Pmin_std)

# --- 相对增强百分比 ---
relative_strength = (Pmax_mean - Pmin_mean) / Pmin_mean * 100
relative_ci = 1.96 * np.sqrt((Pmax_ci / Pmax_mean)**2 + (Pmin_ci / Pmin_mean)**2) * 100

print(f"Pmax mean = {Pmax_mean:.2f} ± {Pmax_ci:.2f}")
print(f"Pmin mean = {Pmin_mean:.2f} ± {Pmin_ci:.2f}")
print(f"Pmax 比 Pmin 强了 {relative_strength:.1f}% ± {relative_ci:.1f}%")

In [ ]:
# Plotting
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
# 加载.rgb文件中的颜色
colorcode = sio.loadmat(str(DATA / "ssta.mat"))['b']
colorcode = colorcode 
custom_cmap = ListedColormap(colorcode)
# 测试色带
data = np.random.rand(8, 8)
plt.imshow(data, cmap=custom_cmap)
plt.colorbar()
plt.show()

In [ ]:
# 创建图形和GridSpec对象
fig = plt.figure(figsize=(14, 7))
fig.set_facecolor([1, 1, 1])  # 设置背景为白色
gs = fig.add_gridspec(1, 7)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.weight'] = 'normal'
plt.rcParams['font.size'] = 16

# 子图1 - ITCZ质心
ax1 = fig.add_subplot(gs[0, 5:])
precession_phases = np.linspace(0, 360, 24)
y_labels = np.arange(0, 360, 15)

# 载入ITCZ数据
barycenter = sio.loadmat(str(DATA / "exp24aprczhixin.mat"))['barycenter']
error24 = sio.loadmat(str(DATA / "exp24aprczhixin.mat"))['error24']
barycenter = np.squeeze(np.vstack((barycenter, barycenter[0, :])))
error24 = np.squeeze(np.vstack((error24, error24[0, :])))

# 画误差棒和填充区域
ax1.errorbar(barycenter, y, xerr=error24, fmt='s', color='r', linewidth=1.5, markersize=3, capsize=3)
ax1.plot(barycenter, y, 'r', linestyle='-', marker='o', linewidth=2, markersize=3, label='IPWP centroid longitude')
lower_bound = barycenter - error24
upper_bound = barycenter + error24
ax1.fill_betweenx(y, lower_bound, upper_bound, color='pink', alpha=0.3, label='95% confidence interval')
ax1.grid(True, linestyle='-.', color='lightgray') 
# 设置y轴
#ax1.set_yticks(np.arange(0, 361, 15))
#ax1.set_ylim([-4, 364])
#ax1.set_yticklabels([])

# 设置x轴
ax1.set_xlabel('Longitude', fontweight='normal', fontsize=16)
ax1.set_xticks(np.arange(120, 145, 5))
ax1.set_xlim([128, 140])
ax1.set_xticklabels(['120°E', '125°E', '130°E', '135°E', '140°E'], fontweight='normal')

# 设置y轴
ax1.set_yticks(np.arange(0, 361, 15))
ax1.set_ylim([-2, 362])
labels = [f'{i}°' if i % 30 == 0 else '' for i in np.arange(0, 361, 15)]
ax1.set_yticklabels(labels, fontweight='normal')
ax1.set_ylabel('Precessional phase', fontweight='normal')
ax1.yaxis.set_label_position("right")   # ✅ 标签移到右侧
ax1.yaxis.tick_right()                  # ✅ 刻度线和标签也移到右侧
ax1.tick_params(axis='y', which='both', direction='in', right=True, left=True)
# 样式美化
# ax1.grid(True, linestyle='-.', color='lightgray')
ax1.tick_params(labelcolor='k', labelsize='medium', width=3)
ax1.spines['top'].set_visible(True)
ax1.spines['right'].set_visible(True)
ax1.spines['bottom'].set_linewidth(1.5)
ax1.spines['left'].set_linewidth(1.5)
ax1.spines['top'].set_linewidth(1.5)
ax1.spines['right'].set_linewidth(1.5)
ax1.tick_params(axis='both', direction='in')
ax1.tick_params(right=True)
# 添加蓝框区域
ax1.axhspan(255, 285, color='cyan', alpha=0.3)
ax1.axhspan(75, 105, color='cyan', alpha=0.3)
ax1.text(128, 379, '(b)', color='black', weight='normal', va='top', fontsize=16)
ax1.legend(
    loc='upper right',
    bbox_to_anchor=(1.05, 0.13),  # ✅ x=1.15 往右移一点
    fontsize=14.5,
    frameon=False
)

# 子图2 - SST std图
ax2 = fig.add_subplot(gs[0, :5])

# contourf绘图
c2 = ax2.contourf(lonstd, y_interp, stdexp, levels=np.arange(0.4, 1.6, 0.005), cmap=custom_cmap, extend='both')

# 设置x轴
ax2.set_xticks(np.arange(160, 281, 20))
ax2.set_xticklabels(['160°E', '180°', '160°W', '140°W', '120°W', '100°W', '80°W'], fontweight='normal')
ax2.set_xlabel('Lontitude', fontweight='normal', fontsize=16)
ax2.set_xlim([159, 280.3])


# 设置y轴
ax2.set_yticks(np.arange(0, 361, 15))
ax2.set_ylim([-2, 362])
labels = [f'{i}°' if i % 30 == 0 else '' for i in np.arange(0, 361, 15)]
ax2.set_yticklabels(labels, fontweight='normal')
ax2.set_ylabel('Precessional phase', fontweight='normal')

# 添加蓝框区域
ax2.add_patch(plt.Rectangle((159.5, 75), 120.5, 30, edgecolor='cyan', linewidth=3, linestyle='--', facecolor='none'))
ax2.add_patch(plt.Rectangle((159.5, 255), 120.5, 30, edgecolor='cyan', linewidth=3, linestyle='--', facecolor='none'))

# 图形样式美化
ax2.grid(False)
ax2.spines['top'].set_visible(True)
ax2.spines['right'].set_visible(True)
ax2.spines['bottom'].set_visible(True)
ax2.spines['left'].set_visible(True)
ax2.spines['bottom'].set_linewidth(1.5)
ax2.spines['left'].set_linewidth(1.5)
ax2.spines['top'].set_linewidth(1.5)
ax2.spines['right'].set_linewidth(1.5)
ax2.tick_params(axis='both', direction='in')
ax2.tick_params(labelcolor='k', labelsize='medium', width=3)
ax2.text(159, 379, '(a)', color='black', weight='normal', va='top', fontsize=16)
ax2.tick_params(axis='y', which='both', direction='in', right=True, left=True)
# 添加 colorbar，使用 add_axes 的方式手动设置位置
ax3 = fig.add_axes([0.055, 0,0.7, 0.2])
ax3.axis('off') 
colorbar = fig.colorbar(c2, ax=ax3, orientation='horizontal', pad=0.2, shrink=0.85, aspect=35)
colorbar.set_label('Standard deviation of SSTA [°C]', fontsize=16, labelpad=-34, fontweight='normal')
colorbar.ax.xaxis.label.set_position((1.31,12))  # 调整标签位置
ticks = np.arange(0.4, 2, 0.4)
colorbar.set_ticks(ticks)
ticklabels = ['{:.1f}'.format(tick) for tick in ticks]
colorbar.set_ticklabels(ticklabels, fontweight='normal')


# 手动调整子图间距（不用 tight_layout 避免冲突）
plt.subplots_adjust(wspace=0.3)
plt.show()
